# Inside BatBot: waveform → spectrogram → model result

[Open in Colab](https://colab.research.google.com/github/Kitware/batbot/blob/main/examples/notebooks/02_spectrogram_walkthrough.ipynb) · [Open Codespaces](https://codespaces.new/Kitware/batbot) · [Quick classification notebook](01_classify_your_files.ipynb)

Use your own WAV and inspect the real processing functions: waveform, STFT, per-frequency normalization/gain, quantization, candidate selection, compressed images, contours/masks, ONNX windows, and final class scores.

A **one-second inspection clip** keeps debugging bounded. Its prediction describes that clip, not the entire source recording. Use notebook 01 for full-file/folder classification. No sample is a scientifically labeled validation set.

## 1. Environment: Codespaces, Colab, or local Jupyter

- **Codespaces:** choose this repository/your fork and the desired branch, let the dev-container setup finish, then select **Python (BatBot)**.
- **Colab:** open this notebook using **File → Open notebook → GitHub** or its Colab link. CPU works immediately; no GPU allocation is required.
- **Local:** install `python -m pip install -e ".[cpu,notebook]"` and select that environment's kernel.

The next cell installs into the active Python kernel when needed. It uses an existing checkout without switching branches or discarding edits; Colab clones a fresh checkout if needed. Before this PR is merged, set `SOURCE_REF` to the branch containing these files. Review code before running it.

Keep `RUNTIME_PROFILE="cpu"` for the quickest first run. For optional Colab GPU use, select a GPU runtime first, then choose `"auto"` (driver-aware), `"cuda12"` (ONNX Runtime 1.26/CUDA 12), or `"cuda13"` (1.31/CUDA 13). GPU dependencies are large, a compatible driver/device is required, and unavailable explicit providers fail clearly. Start a **fresh runtime** before changing CPU/GPU profiles; do not install both ONNX distributions. Native macOS can select `"coreml"`.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

SOURCE_REF = "main"  # Before merge, use the branch containing these notebooks.
RUNTIME_PROFILE = "cpu"  # cpu, auto, cuda12, cuda13, or coreml (native macOS)
REPO_URL = "https://github.com/Kitware/batbot.git"

override = os.environ.get("BATBOT_NOTEBOOK_REPO")
REPO = Path(override).resolve() if override else next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "pyproject.toml").is_file() and (p / "batbot").is_dir()), None
)
if REPO is None:
    REPO = Path.cwd() / "batbot-tutorial-source"
    if not REPO.exists():
        clone_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--no-checkout", REPO_URL, str(REPO)],
            env=clone_env,
        )
        # Fetch explicitly so SOURCE_REF can be a branch, tag, or exact commit SHA.
        subprocess.check_call(
            ["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", SOURCE_REF],
            env=clone_env,
        )
        subprocess.check_call(
            ["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], env=clone_env
        )
helper = REPO / "examples/notebooks/notebook_utils.py"
if not helper.is_file():
    raise RuntimeError("This checkout lacks the tutorial helpers. Select the notebook's branch/tag.")
spec = importlib.util.spec_from_file_location("batbot_notebook_utils", helper)
nb = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nb)
PROFILE, PROVIDER = nb.prepare_environment(REPO, RUNTIME_PROFILE)
print(f"Kernel: {sys.executable} (Python {sys.version.split()[0]})")
print(f"Using checkout: {REPO}. SOURCE_REF applies only when cloning a new checkout.")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from batbot import classifier

get_ipython().run_line_magic("matplotlib", "inline")

## 2. Select/upload a WAV and choose a small inspection interval

Drag WAVs into `data/input/` in Codespaces/JupyterLab, or use Colab's opt-in upload/Drive options below. Set `WAV_INDEX`, `START_SECONDS`, and `CLIP_SECONDS` to inspect a call of interest. Keep the original file unchanged; the clip is saved under this unique run's output folder. With no matching WAV, the public example is used and clearly identified.

Debugging can generate many files and be much slower than a spectrogram preview. Start at one second; the helper caps inspection clips at ten seconds. Low-rate audio cannot supply ultrasonic frequencies that were never recorded.

In [ ]:
import tempfile

DATA_DIR = nb.native_path(os.environ.get("BATBOT_NOTEBOOK_INPUT_DIR", REPO / "data/input"))
OUTPUT_ROOT = nb.native_path(os.environ.get("BATBOT_NOTEBOOK_OUTPUT_DIR", REPO / "artifacts/notebooks"))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="run-", dir=OUTPUT_ROOT))
UPLOAD_FILES = False  # Colab: set True to open a multi-file picker.
MOUNT_DRIVE = False   # Colab: set True, then point DATA_DIR at your permitted Drive folder.
USE_EXAMPLE_IF_EMPTY = True
print("Input:", DATA_DIR)
print("Outputs:", RUN_DIR)

WAV_INDEX = 0
START_SECONDS = 0.0
CLIP_SECONDS = float(os.environ.get("BATBOT_NOTEBOOK_CLIP_SECONDS", "1.0"))

In [ ]:
if MOUNT_DRIVE or UPLOAD_FILES:
    try:
        from google.colab import drive, files
    except ImportError as error:
        raise RuntimeError("Outside Colab, upload using the file explorer/JupyterLab sidebar.") from error
    if MOUNT_DRIVE:
        drive.mount("/content/drive")
    if UPLOAD_FILES:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        files.upload(target_dir=str(DATA_DIR))

In [ ]:
wav_inputs = nb.choose_inputs(DATA_DIR, REPO, OUTPUT_ROOT / "cache", maximum=8,
                              allow_example=USE_EXAMPLE_IF_EMPTY, only_wav=True)
source = wav_inputs[WAV_INDEX]
waveform, sample_rate, clip = nb.crop_recording(
    source, RUN_DIR / "inspection.wav", start=START_SECONDS, seconds=CLIP_SECONDS,
)
display(pd.Series({"source": str(source), "clip": str(clip), "sample_rate_hz": sample_rate,
                   "clip_seconds": len(waveform) / sample_rate, "start_seconds": START_SECONDS}))

## 3. Waveform, STFT, gain and quantization

These are **BatBot's actual functions**, not an independently reimplemented preprocessing pipeline:

1. Load mono audio and resample to **250 kHz**.
2. Use a **512-point FFT**, a **256-sample Blackman–Harris window**, and a **16-sample hop**.
3. Select approximately **5–120 kHz**; suppress nonphysical bands above the source Nyquist limit.
4. Subtract each frequency's median, shift the peak, apply the current **120 dB gain** and optional MAD-based autogain, and clip.
5. Quantize to **16-bit** for the full preprocessing mode and flip vertically when writing images.

The displayed time axis is in the inspection clip. Preview columns may be subsampled for display only; processing and inference still use the original arrays. After gain/quantization, values are intensities—not physical dB measurements.

In [ ]:
figure, stage_info = nb.plot_stages(clip, waveform, sample_rate, RUN_DIR / "stages.png")
display(figure)
plt.close(figure)
display(pd.Series(stage_info))

## 4. Run full processing with debugging enabled

The pipeline selects candidate time ranges, tightens/refines contours, and computes metadata. It writes full images, compressed chirp views, and—when available—masks/contours. The classifier currently consumes the **full uncompressed JPEG outputs**. Compressed/masked/debug views are useful for diagnosis but are **not substituted as model inputs** in this notebook.

`fast_mode=True` is a separate, reduced mode with different candidate settings and 8-bit data; it is not the WAV classifier default. This walkthrough uses `fast_mode=False`.

Re-running this cell replaces only its run-specific debug folder. It never deletes or overwrites original input files.

In [ ]:
from batbot import spectrogram

full_images, compressed_images, metadata_path, metadata = spectrogram.compute(
    clip, output_folder=RUN_DIR / "processing",
    fast_mode=False, debug=True, force_overwrite=True, quiet=True,
    plot_uncompressed_amplitude=True, include_original_sr=False,
)
debug_dir = RUN_DIR / "processing" / f"batbot.debug.{clip.name}"
print("Full model-input images:", full_images)
print("Compressed inspection images:", compressed_images)
print("Metadata:", metadata_path)
print("Selected debug files:", [path.name for path in sorted(debug_dir.iterdir())[:12]])

In [ ]:
inspection_images = [
    *full_images[:1], *compressed_images[:1],
    *metadata["spectrogram"].get("mask.path", [])[:1],
    *metadata["spectrogram"].get("masked.path", [])[:1],
]
display(nb.show_images(inspection_images, RUN_DIR / "processing_comparison.png", limit=4,
                       title="Full input, compressed inspection view, mask and masked view"))
plt.close("all")
selected_debug = [
    debug_dir / "candidates.png",
    debug_dir / "spectrogram.compressed.threshold.jet.tif",
    debug_dir / "spectrogram.compressed.keypoints.tif",
    *sorted(debug_dir.glob("contour.*.png"))[:1],
]
display(nb.show_images([path for path in selected_debug if path.is_file()],
                       RUN_DIR / "debug_gallery.png", limit=4,
                       title="Selected candidate/contour diagnostics (not model inputs)"))
plt.close("all")

## 5. Inspect timing metadata

Full images preserve acquisition time. **Compressed images concatenate selected ranges**, so their pixel positions no longer represent a continuous recording-time axis. Segment timestamps in JSON refer to the clip; add `START_SECONDS` to relate them to the original recording. Detected segments are not verified bat calls, and overlapping classifier windows are not individual animals.

Some recordings have no refined contours/masks/compressed views. That is a useful diagnostic, not an automatic `NOISE` label. The full-image model can still return scores.

In [ ]:
segments = pd.DataFrame(metadata.get("segments", []))
display(pd.Series({
    "segments": len(segments), "duration_ms": metadata["duration.ms"],
    "analysis_sample_rate": metadata["sr.hz"], "image_sizes": metadata["size"],
}))
display(segments.iloc[:10, :8] if not segments.empty else pd.DataFrame({"note": ["No refined segments"]}))

## 6. See exactly what ONNX receives

Use `classifier.pre` on the **first full-image chunk**. It rescales height to 224, scales width by half the vertical ratio, and creates overlapping 224-pixel windows at stride 100. Narrow images are padded. The input tensor is contiguous **uint8 BGR**, not manually normalized RGB floats.

The previews convert BGR to RGB for plotting only. Window index identifies an overlapping region of the image—not a call identifier or an independent biological observation.

In [ ]:
if not full_images:
    raise RuntimeError("No full spectrogram was created. Inspect the recording and pipeline logs.")
windows, model_key = next(classifier.pre([full_images[0]]))
display(pd.Series({"model": model_key, "tensor_shape": windows.shape, "dtype": str(windows.dtype)}))
count = min(6, len(windows))
figure, axes = plt.subplots(1, count, figsize=(2.5 * count, 3), squeeze=False, layout="constrained")
for index, axis in enumerate(axes[0]):
    axis.imshow(windows[index, :, :, ::-1])  # Display conversion only.
    axis.set_title(f"Window {index}")
    axis.set_axis_off()
figure.savefig(RUN_DIR / "model_windows.png", dpi=140)
display(figure)
plt.close(figure)

## 7. Window scores, averaging and final clip classification

The heatmap exposes per-window model scores for the first image's strongest classes. Averaging those scores must match `runner.classify` for that image. A WAV spanning multiple chunks combines image means using their window counts.

Finally, call the **public WAV API** on the inspection clip. This regenerates normal, non-debug full images and returns one clip-level result. This extra pass deliberately verifies the real public workflow rather than treating tutorial-specific image manipulation as production inference.

In [ ]:
runner = classifier.Classifier(providers=[PROVIDER], batch_size=10, num_workers=1)
per_window = nb.window_scores(runner, windows)
image_result = runner.classify([full_images[0]], progress=False)[0]
np.testing.assert_allclose(
    per_window.mean(axis=0),
    [image_result["scores"][label] for label in runner.config.classes],
    rtol=1e-5, atol=1e-7,
)
strongest = np.argsort(per_window.mean(axis=0))[-6:][::-1]
figure, axis = plt.subplots(figsize=(10, 4), layout="constrained")
artist = axis.imshow(per_window[:, strongest].T, aspect="auto", cmap="viridis", vmin=0, vmax=1)
axis.set_yticks(range(len(strongest)), [runner.config.classes[index] for index in strongest])
axis.set(xlabel="Overlapping window index (first full image)", ylabel="Class",
         title="Per-window scores; their mean equals the image result")
figure.colorbar(artist, ax=axis, label="Model score")
figure.savefig(RUN_DIR / "window_scores.png", dpi=140)
display(figure)
plt.close(figure)

result = runner.classify_wav(
    clip, output_folder=RUN_DIR / "inference", keep_spectrograms=True, progress=False,
)
report = {"results": [result], "summary": classifier.summarize([result])}
table, scores = nb.result_tables(report)
display(table)
display(pd.DataFrame(result["top"]))
display(nb.plot_results(report, RUN_DIR / "clip_classification.png"))
plt.close("all")

## 8. Export the experiment and take the next small step

Save the source/clip interval, algorithm outputs, model hash, runtime, scores, and charts. The ZIP **does contain the short inspection WAV** and derived debug data; share it only with permission. Default runs never upload data or submit anything to NABat.

Try another interval or a different recording first. Preserve one baseline before modifying a preprocessing parameter. A preprocessing/model change is a research hypothesis: evaluate it on reviewed, held-out data and discuss it in a draft BatBot/BatAI pull request.

In [ ]:
provenance = nb.export_report(report, runner, RUN_DIR)
(RUN_DIR / "inspection_settings.json").write_text(
    json.dumps({"source": str(source), "start_seconds": START_SECONDS,
                             "clip_seconds": CLIP_SECONDS, "runtime_profile": PROFILE,
                             "debug_fast_mode": False, **provenance}, indent=2), encoding="utf8"
)
archive = nb.archive_outputs(RUN_DIR)
print("Experiment archive:", archive)
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive))

## Collaborate responsibly

Code contributions and reproducible prototypes are welcome in [BatBot](https://github.com/Kitware/batbot) and [Kitware BatAI](https://github.com/Kitware/batai). **Monitoring data contributions belong in the [USGS NABat program](https://www.usgs.gov/centers/fort-collins-science-center/science/north-american-bat-monitoring-program-nabat) database via the [Partner Portal](https://sciencebase.usgs.gov/nabat/), not GitHub.** Follow [NABat's current upload guidance](https://www.nabatmonitoring.org/upload-data).

These exports are exploratory results, not NABat-ready submissions, verified observations, or evidence of program approval. Share only permitted, non-sensitive information. Reports/debug images can contain paths and recording details. Clear notebook outputs before a pull request; never commit private audio, locations, credentials, or generated archives.